In [1]:
import os
import joblib
import pandas as pd

In [ ]:
X_TRAIN_CLEANED_PATH = "../data/cleaned/features_train_raw_cleaned_for_predict.csv"
X_TEST_CLEANED_PATH = "../data/cleaned/features_test_raw_cleaned_for_predict.csv"

MODEL_DIR_PATH = "../models"

**geo_mapping file for not make user digit province, region, lat and lon**

In [7]:
# unite cleaned features datasets

train_df = pd.read_csv(X_TRAIN_CLEANED_PATH)
test_df = pd.read_csv(X_TEST_CLEANED_PATH)

full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

In [8]:
geo_cols = ['postal_code', 'province', 'region', 'latitude', 'longitude', 'property_subtype']

geo_mapping = full_df[geo_cols].drop_duplicates(subset=['postal_code']).set_index('postal_code').to_dict(orient='index')

In [ ]:
joblib.dump(geo_mapping,  os.path.join(MODEL_DIR_PATH, 'geo_mapping.joblib'))
print(f"✅ Joblib file for geo_mapping successfully generated. {len(geo_mapping)} unique posta codes covered.\nAdd the joblib file in the api directory of project 06.")

✅ Joblib file for geo_mapping successfully generated. 954 unique posta codes covered.
Add the joblib file in the api directory of project 06.


**Refactoring of `handle_missing_values()` for deployment**

In [ ]:
# calculate global medians to be saved for production preprocessing (proj 06)

# calculate medians for lat and lon grouping them by 'province'
lat_median_by_province = df.groupby('province')['latitude'].median()
lon_median_by_province = df.groupby('province')['longitude'].median()

# calculate national medians (for lat and lon, just in case 'province' is missing)
national_defaults_medians = {
    'latitude': df['latitude'].median(),
    'longitude': df['longitude'].median(),
    'living_area_ratio': df['living_area_ratio'].median(),
    'bedroom_density': df['bedroom_density'].median(),
    'urban_density': df['urban_density'].median() if 'urban_density' in df.columns else 300.0,
    'property_age': df['property_age'].median() if 'building_year' in df.columns else 40.0
}

deployment_medians = {
    'by_province': {
        'latitude': lat_median_by_province,
        'longitude': lon_median_by_province
    },
    'national': national_defaults_medians
}

joblib.dump(deployment_medians, "../models/global_medians.joblib")
print("✅ global_medians.joblib suiccessfully saved in models dir.")